# ЛР1 — Основы Data Science
## Ames Housing — задания для самостоятельной работы

**Цель:** выполнить задания для самостоятельной работы по методичке на базе Ames Housing Dataset.

В работе рассматриваются:
- дополнительный Feature Engineering;
- обработка пропусков;
- Linear Regression, Ridge и Lasso;
- анализ остатков;
- 10-fold cross-validation.

## 1. Импорт библиотек

Набор библиотек соответствует методичке.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.stats import skew

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 7)
plt.rcParams["font.size"] = 10

print("Все библиотеки успешно импортированы!")

## 2. Загрузка данных

Если в `data/` находится полный `train.csv`, используется он.
Иначе используется демонстрационный набор из методички.

In [ ]:
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "data").exists():
    project_root = Path.cwd().parent

full_train = project_root / "data" / "train.csv"
demo_train = project_root / "data" / "ames_demo.csv"

if full_train.exists():
    df = pd.read_csv(full_train)
    print(f"Загружен полный датасет: {full_train}")
else:
    df = pd.read_csv(demo_train)
    print(f"Загружен демонстрационный датасет: {demo_train}")

print(f"Размер данных: {df.shape}")
display(df.head())

## 3. Первичный анализ данных

In [ ]:
print("Размер датасета:", df.shape)
print("\nТипы данных:")
display(df.dtypes.to_frame("dtype"))

print("\nОсновная статистика:")
display(df.describe(include="all").T)

print("\nКоличество пропусков:")
display(df.isnull().sum()[df.isnull().sum() > 0].sort_values(ascending=False))

## 4. Анализ пропущенных значений

In [ ]:
def analyze_missing_values(dataframe):
    missing_data = dataframe.isnull().sum()
    missing_data = missing_data[missing_data > 0].sort_values(ascending=False)
    missing_percent = (missing_data / len(dataframe)) * 100

    return pd.DataFrame({
        "Количество пропусков": missing_data,
        "Процент пропусков": missing_percent
    })

missing_analysis = analyze_missing_values(df)

if len(missing_analysis) > 0:
    display(missing_analysis)

    fig, axes = plt.subplots(2, 1, figsize=(12, 8))
    missing_analysis["Количество пропусков"].plot(kind="bar", ax=axes[0])
    axes[0].set_title("Количество пропущенных значений")
    axes[0].set_ylabel("Количество")

    missing_analysis["Процент пропусков"].plot(kind="bar", ax=axes[1])
    axes[1].set_title("Процент пропущенных значений")
    axes[1].set_ylabel("Процент (%)")

    plt.tight_layout()
    plt.show()

    print(f"Признаков с пропусками: {len(missing_analysis)}")
else:
    print("Пропущенных значений не найдено.")

## 5. Обработка пропусков — текущий метод из методички

In [ ]:
def handle_missing_values(dataframe):
    dataframe = dataframe.copy()

    # Отсутствующий объект трактуем как отсутствие объекта
    categorical_na_as_none = ["Alley", "MasVnrType", "BsmtQual", "GarageType"]

    for feature in categorical_na_as_none:
        if feature in dataframe.columns:
            dataframe[feature] = dataframe[feature].fillna("None")

    # Отсутствующий объект -> площадь 0
    if "MasVnrArea" in dataframe.columns:
        dataframe["MasVnrArea"] = dataframe["MasVnrArea"].fillna(0)

    # Остальные числовые признаки -> медиана
    numeric_features = dataframe.select_dtypes(include=[np.number]).columns
    for feature in numeric_features:
        if dataframe[feature].isnull().sum() > 0:
            dataframe[feature] = dataframe[feature].fillna(dataframe[feature].median())

    # Остальные категориальные признаки -> мода
    categorical_features = dataframe.select_dtypes(include=["object"]).columns
    for feature in categorical_features:
        if dataframe[feature].isnull().sum() > 0:
            dataframe[feature] = dataframe[feature].fillna(dataframe[feature].mode()[0])

    return dataframe

df_processed = handle_missing_values(df)

print("Оставшиеся пропуски:", df_processed.isnull().sum().sum())

## 6. Feature Engineering из основной части методички

In [ ]:
def create_new_features(dataframe):
    df_new = dataframe.copy()
    current_year = 2023

    if "YearBuilt" in df_new.columns:
        df_new["HouseAge"] = current_year - df_new["YearBuilt"]

    if "YearRemodAdd" in df_new.columns:
        df_new["YearsSinceRemod"] = current_year - df_new["YearRemodAdd"]

    if all(col in df_new.columns for col in ["TotalBsmtSF", "GrLivArea"]):
        df_new["TotalSF"] = df_new["TotalBsmtSF"] + df_new["GrLivArea"]

    if "FullBath" in df_new.columns:
        df_new["TotalBathrooms"] = (
            df_new["FullBath"] + df_new.get("HalfBath", 0) * 0.5
        )

    if all(col in df_new.columns for col in ["GarageArea", "GrLivArea"]):
        df_new["GarageRatio"] = df_new["GarageArea"] / (df_new["GrLivArea"] + 1)

    if "OverallQual" in df_new.columns:
        df_new["QualityCategory"] = pd.cut(
            df_new["OverallQual"],
            bins=[0, 4, 7, 10],
            labels=["Low", "Medium", "High"]
        )

    if all(col in df_new.columns for col in ["OverallQual", "GrLivArea"]):
        df_new["IsPremium"] = (
            (df_new["OverallQual"] >= 8)
            & (df_new["GrLivArea"] > df_new["GrLivArea"].quantile(0.75))
        ).astype(int)

    return df_new

df_processed = create_new_features(df_processed)
print("Размер после Feature Engineering:", df_processed.shape)
display(df_processed.head())

## 7. Кодирование категориальных признаков

In [ ]:
def encode_categorical_features(dataframe):
    df_encoded = dataframe.copy()

    ordinal_features = {
        "OverallQual": list(range(1, 11)),
        "OverallCond": list(range(1, 11)),
        "BsmtQual": ["None", "Po", "Fa", "TA", "Gd", "Ex"],
        "KitchenQual": ["Po", "Fa", "TA", "Gd", "Ex"],
        "QualityCategory": ["Low", "Medium", "High"]
    }

    for feature, order in ordinal_features.items():
        if feature in df_encoded.columns:
            mapping = {value: i for i, value in enumerate(order)}
            df_encoded[f"{feature}_encoded"] = df_encoded[feature].map(mapping)
            df_encoded[f"{feature}_encoded"] = (
                df_encoded[f"{feature}_encoded"].fillna(0)
            )

    categorical_features = df_encoded.select_dtypes(include=["object"]).columns.tolist()
    nominal_features = [
        f for f in categorical_features
        if f not in ordinal_features
    ]

    for feature in nominal_features:
        unique_count = df_encoded[feature].nunique()
        if unique_count <= 10:
            dummies = pd.get_dummies(
                df_encoded[feature],
                prefix=feature,
                drop_first=True
            )
            df_encoded = pd.concat([df_encoded, dummies], axis=1)

    df_encoded = df_encoded.select_dtypes(exclude=["object"])
    return df_encoded

df_encoded = encode_categorical_features(df_processed)
print("Количество признаков после кодирования:", df_encoded.shape[1])

## 8. Подготовка признаков и логарифмическое преобразование

In [ ]:
def prepare_features(dataframe, target_column="SalePrice"):
    X = dataframe.drop([target_column, "Id"], axis=1, errors="ignore").copy()
    y = dataframe[target_column].copy()

    target_skewness = skew(y)

    if abs(target_skewness) > 0.5:
        y_transformed = np.log1p(y)
        print(
            f"SalePrice преобразован: skewness "
            f"{target_skewness:.3f} -> {skew(y_transformed):.3f}"
        )
    else:
        y_transformed = y
        print(f"SalePrice не требует преобразования: skewness={target_skewness:.3f}")

    numeric_features = X.select_dtypes(include=[np.number]).columns
    skewed_features = []

    for feature in numeric_features:
        if X[feature].nunique() > 10:
            feature_skewness = skew(X[feature])
            if abs(feature_skewness) > 0.75:
                skewed_features.append(feature)

    X_transformed = X.copy()

    for feature in skewed_features:
        # В методичке log1p применяется к скошенным числовым признакам
        X_transformed[feature] = np.log1p(X_transformed[feature].clip(lower=0))

    print("Сильно скошенные признаки:", skewed_features)
    return X_transformed, y_transformed, skewed_features

X_processed, y_processed, skewed_features = prepare_features(df_encoded)

print("X:", X_processed.shape)
print("y:", y_processed.shape)

## 9. Train/Test и базовые модели

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_processed,
    y_processed,
    test_size=0.2,
    random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Размер обучающей выборки:", X_train.shape)
print("Размер тестовой выборки:", X_test.shape)

In [ ]:
def train_and_evaluate_models(
    X_train, X_test, y_train, y_test,
    X_train_scaled, X_test_scaled
):
    models = {}
    results = {}

    # Linear Regression
    lr = LinearRegression()
    lr.fit(X_train, y_train)
    lr_pred = lr.predict(X_test)

    models["Linear"] = lr
    results["Linear"] = {
        "predictions": lr_pred,
        "mae": mean_absolute_error(y_test, lr_pred),
        "mse": mean_squared_error(y_test, lr_pred),
        "r2": r2_score(y_test, lr_pred)
    }

    # Ridge
    ridge_params = {"alpha": [0.1, 1.0, 10.0, 100.0, 1000.0]}
    ridge_grid = GridSearchCV(
        Ridge(),
        ridge_params,
        cv=5,
        scoring="r2"
    )
    ridge_grid.fit(X_train_scaled, y_train)

    best_ridge = ridge_grid.best_estimator_
    ridge_pred = best_ridge.predict(X_test_scaled)

    models["Ridge"] = best_ridge
    results["Ridge"] = {
        "predictions": ridge_pred,
        "mae": mean_absolute_error(y_test, ridge_pred),
        "mse": mean_squared_error(y_test, ridge_pred),
        "r2": r2_score(y_test, ridge_pred),
        "best_alpha": ridge_grid.best_params_["alpha"]
    }

    # Lasso
    lasso_params = {"alpha": [0.001, 0.01, 0.1, 1.0, 10.0]}
    lasso_grid = GridSearchCV(
        Lasso(max_iter=10000),
        lasso_params,
        cv=5,
        scoring="r2"
    )
    lasso_grid.fit(X_train_scaled, y_train)

    best_lasso = lasso_grid.best_estimator_
    lasso_pred = best_lasso.predict(X_test_scaled)

    models["Lasso"] = best_lasso
    results["Lasso"] = {
        "predictions": lasso_pred,
        "mae": mean_absolute_error(y_test, lasso_pred),
        "mse": mean_squared_error(y_test, lasso_pred),
        "r2": r2_score(y_test, lasso_pred),
        "best_alpha": lasso_grid.best_params_["alpha"],
        "n_features_used": int(np.sum(best_lasso.coef_ != 0))
    }

    return models, results

models, results = train_and_evaluate_models(
    X_train, X_test, y_train, y_test,
    X_train_scaled, X_test_scaled
)

results_table = pd.DataFrame({
    name: {
        "MAE": value["mae"],
        "MSE": value["mse"],
        "R²": value["r2"]
    }
    for name, value in results.items()
}).T

display(results_table)

# Задание 1. Дополнительный Feature Engineering

По методичке необходимо создать:
- `PricePerSqFt`;
- `AgeCategory`;
- `HasGarage`.

In [ ]:
def create_additional_features(dataframe):
    df_new = dataframe.copy()

    # Цена за квадратный фут.
    # Этот признак используется только для анализа,
    # потому что SalePrice является целевой переменной.
    if "SalePrice" in df_new.columns and "GrLivArea" in df_new.columns:
        df_new["PricePerSqFt"] = (
            df_new["SalePrice"] / (df_new["GrLivArea"] + 1)
        )

    # Категория возраста дома
    if "HouseAge" in df_new.columns:
        df_new["AgeCategory"] = pd.cut(
            df_new["HouseAge"],
            bins=[0, 10, 30, 100],
            labels=["New", "Medium", "Old"]
        )

    # Наличие гаража
    if "GarageArea" in df_new.columns:
        df_new["HasGarage"] = (
            df_new["GarageArea"] > 0
        ).astype(int)

    return df_new

df_additional = create_additional_features(df_processed)

print("Новые признаки:")
display(
    df_additional[
        [c for c in ["PricePerSqFt", "HouseAge", "AgeCategory", "HasGarage"]
         if c in df_additional.columns]
    ].head(10)
)

print("\nРаспределение AgeCategory:")
display(df_additional["AgeCategory"].value_counts(dropna=False))

print("\nРаспределение HasGarage:")
display(df_additional["HasGarage"].value_counts(dropna=False))

### Анализ PricePerSqFt

`PricePerSqFt` нельзя передавать в модель прогнозирования `SalePrice`, поскольку он вычислен с использованием самой `SalePrice`. Это демонстрационный признак из задания методички.

In [ ]:
if "PricePerSqFt" in df_additional.columns:
    print(
        f"Корреляция PricePerSqFt с SalePrice: "
        f"{df_additional['PricePerSqFt'].corr(df_additional['SalePrice']):.3f}"
    )

    plt.figure(figsize=(10, 6))
    plt.scatter(
        df_additional["GrLivArea"],
        df_additional["PricePerSqFt"],
        alpha=0.7
    )
    plt.xlabel("GrLivArea")
    plt.ylabel("PricePerSqFt")
    plt.title("Цена за квадратный фут")
    plt.show()

### Проверка влияния `HasGarage`

Для проверки влияния признака можно сравнить среднюю цену домов с гаражом и без гаража.

In [ ]:
garage_comparison = (
    df_additional
    .groupby("HasGarage")["SalePrice"]
    .agg(["mean", "median", "count"])
)

display(garage_comparison)

# Задание 2. Улучшение обработки пропусков

Используем `KNNImputer` для всех числовых признаков с пропусками и сравним его с текущим подходом.

В соответствии с методичкой KNN используется для числовых признаков.

In [ ]:
# Для сравнения берём исходный df до обработки пропусков.
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

if "Id" in numeric_cols:
    numeric_cols.remove("Id")

print("Числовые признаки:")
print(numeric_cols)

print("\nПропуски до обработки:")
display(df[numeric_cols].isnull().sum()[df[numeric_cols].isnull().sum() > 0])

In [ ]:
# Текущий метод: медиана для числовых признаков
df_median = df.copy()

for feature in numeric_cols:
    if df_median[feature].isnull().sum() > 0:
        df_median[feature] = df_median[feature].fillna(
            df_median[feature].median()
        )

# KNNImputer
df_knn = df.copy()

knn_imputer = KNNImputer(n_neighbors=5)
df_knn[numeric_cols] = knn_imputer.fit_transform(
    df_knn[numeric_cols]
)

print("Пропуски после медианного метода:",
      df_median[numeric_cols].isnull().sum().sum())

print("Пропуски после KNNImputer:",
      df_knn[numeric_cols].isnull().sum().sum())

if "LotFrontage" in df.columns:
    comparison = pd.DataFrame({
        "Исходное": df["LotFrontage"],
        "Медиана": df_median["LotFrontage"],
        "KNN": df_knn["LotFrontage"]
    })

    display(comparison)

### Сравнение корреляции для LotFrontage

Это дополнительная проверка, аналогичная демонстрации из методички.

In [ ]:
if "LotFrontage" in df.columns and df["LotFrontage"].isnull().sum() > 0:
    print("Корреляция LotFrontage с SalePrice:")
    print(f"Медиана: {df_median['LotFrontage'].corr(df_median['SalePrice']):.3f}")
    print(f"KNN:     {df_knn['LotFrontage'].corr(df_knn['SalePrice']):.3f}")

### Сравнение качества моделей после KNNImputer

Чтобы сравнение было корректным, повторяем тот же preprocessing и тот же набор моделей.

In [ ]:
def build_model_data(dataframe, imputation="median"):
    data = dataframe.copy()

    # Специальные пропуски
    for feature in ["Alley", "MasVnrType", "BsmtQual", "GarageType"]:
        if feature in data.columns:
            data[feature] = data[feature].fillna("None")

    if "MasVnrArea" in data.columns:
        data["MasVnrArea"] = data["MasVnrArea"].fillna(0)

    numeric_features = data.select_dtypes(include=[np.number]).columns.tolist()
    numeric_without_target = [
        c for c in numeric_features
        if c not in ["Id", "SalePrice"]
    ]

    if imputation == "knn":
        imputer = KNNImputer(n_neighbors=5)
        data[numeric_without_target] = imputer.fit_transform(
            data[numeric_without_target]
        )
    else:
        for feature in numeric_without_target:
            if data[feature].isnull().sum() > 0:
                data[feature] = data[feature].fillna(data[feature].median())

    # Остальные категориальные -> мода
    categorical_features = data.select_dtypes(include=["object"]).columns
    for feature in categorical_features:
        if data[feature].isnull().sum() > 0:
            data[feature] = data[feature].fillna(data[feature].mode()[0])

    data = create_new_features(data)
    data = encode_categorical_features(data)

    X, y, _ = prepare_features(data)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    _, result = train_and_evaluate_models(
        X_train, X_test, y_train, y_test,
        X_train_scaled, X_test_scaled
    )

    return result

median_results = build_model_data(df, "median")
knn_results = build_model_data(df, "knn")

imputation_comparison = pd.DataFrame({
    "Median": {
        model: values["r2"] for model, values in median_results.items()
    },
    "KNN": {
        model: values["r2"] for model, values in knn_results.items()
    }
})

display(imputation_comparison)

best_median = imputation_comparison["Median"].max()
best_knn = imputation_comparison["KNN"].max()

print(f"Лучший R² при медианной обработке: {best_median:.4f}")
print(f"Лучший R² при KNNImputer: {best_knn:.4f}")

if best_knn > best_median:
    print("Вывод: KNNImputer показал лучшее качество на тестовой выборке.")
elif best_knn < best_median:
    print("Вывод: текущий метод показал лучшее качество на тестовой выборке.")
else:
    print("Вывод: качество методов оказалось одинаковым.")

# Задание 3. Анализ остатков

Остаток определяется как:

**residual = фактическое значение − предсказанное значение**

Проверяем:
1. residuals vs predicted;
2. Q-Q plot;
3. наличие систематических паттернов.

In [ ]:
residuals = {}

for model_name, model_result in results.items():
    residuals[model_name] = (
        y_test.values - model_result["predictions"]
    )

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, model_name in zip(axes, residuals.keys()):
    predicted = results[model_name]["predictions"]
    resid = residuals[model_name]

    ax.scatter(predicted, resid, alpha=0.8)
    ax.axhline(0, linestyle="--")
    ax.set_title(f"{model_name}: остатки vs предсказание")
    ax.set_xlabel("Предсказанные значения")
    ax.set_ylabel("Остатки")

plt.tight_layout()
plt.show()

### Q-Q plots для всех моделей

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, model_name in zip(axes, residuals.keys()):
    stats.probplot(
        residuals[model_name],
        dist="norm",
        plot=ax
    )
    ax.set_title(f"Q-Q plot: {model_name}")

plt.tight_layout()
plt.show()

In [ ]:
print("Среднее и стандартное отклонение остатков:")

for model_name, resid in residuals.items():
    print(
        f"{model_name}: "
        f"mean={np.mean(resid):.4f}, "
        f"std={np.std(resid):.4f}"
    )

### Вывод по остаткам

При хорошем регрессионном моделировании остатки должны быть:
- примерно симметричны вокруг нуля;
- не иметь выраженной зависимости от предсказанного значения;
- не образовывать явных кривых или веерообразных структур;
- на Q-Q plot располагаться приблизительно вдоль диагонали.

На небольшом демонстрационном датасете из 20 строк выводы по графикам следует считать предварительными, поскольку размер тестовой выборки очень мал.

# Задание 4. 10-fold Cross-Validation

Применяем 10-fold cross-validation ко всем трём моделям и сравниваем:
- среднее R²;
- стандартное отклонение;
- 95% доверительный интервал.

Это соответствует структуре проверки из методички.

In [ ]:
def cross_validate_models(models, X, y, cv=10):
    cv_results = {}

    print("РЕЗУЛЬТАТЫ 10-FOLD КРОСС-ВАЛИДАЦИИ")
    print("=" * 50)

    for model_name, model in models.items():
        if model_name == "Linear":
            scores = cross_val_score(
                model,
                X,
                y,
                cv=cv,
                scoring="r2"
            )
        else:
            scaler_cv = StandardScaler()
            X_scaled = scaler_cv.fit_transform(X)

            scores = cross_val_score(
                model,
                X_scaled,
                y,
                cv=cv,
                scoring="r2"
            )

        mean_score = scores.mean()
        std_score = scores.std()
        ci_low = mean_score - 2 * std_score
        ci_high = mean_score + 2 * std_score

        cv_results[model_name] = {
            "Mean R²": mean_score,
            "Std R²": std_score,
            "CI 95% low": ci_low,
            "CI 95% high": ci_high,
            "Scores": scores
        }

        print(f"{model_name}:")
        print(f"  Среднее R²: {mean_score:.4f}")
        print(f"  Стд. отклонение: {std_score:.4f}")
        print(f"  95% ДИ: [{ci_low:.4f}, {ci_high:.4f}]")
        print()

    return cv_results

cv_results = cross_validate_models(
    models,
    X_processed,
    y_processed,
    cv=10
)

In [ ]:
cv_table = pd.DataFrame({
    model: {
        "Среднее R²": value["Mean R²"],
        "Стд. отклонение": value["Std R²"],
        "95% CI low": value["CI 95% low"],
        "95% CI high": value["CI 95% high"]
    }
    for model, value in cv_results.items()
})

display(cv_table)

# Стабильность: меньший std означает более стабильные результаты
most_stable_model = cv_table["Стд. отклонение"].idxmin()
best_cv_model = cv_table["Среднее R²"].idxmax()

print(f"Наиболее стабильная модель: {most_stable_model}")
print(f"Лучшее среднее R² в CV: {best_cv_model}")

## Сравнение train/test и 10-fold CV

In [ ]:
comparison = pd.DataFrame({
    "Train/Test R²": {
        model: results[model]["r2"]
        for model in results
    },
    "CV mean R²": {
        model: cv_results[model]["Mean R²"]
        for model in cv_results
    },
    "CV std": {
        model: cv_results[model]["Std R²"]
        for model in cv_results
    }
})

display(comparison)

# Итоговые выводы

## Задание 1
Созданы три дополнительных признака:
- `PricePerSqFt` — цена за квадратный фут;
- `AgeCategory` — категория возраста дома;
- `HasGarage` — наличие гаража.

`PricePerSqFt` не используется как входной признак модели, поскольку вычисляется через `SalePrice`.

## Задание 2
`KNNImputer` применён ко всем числовым признакам с пропусками. Качество сравнено с текущим методом заполнения пропусков.

Лучший вариант определяется по R² тестовой выборки в таблице выше.

## Задание 3
Для Linear Regression, Ridge и Lasso построены графики остатков и Q-Q plots. По ним можно оценить наличие систематических ошибок и отклонений от нормальности.

## Задание 4
Для всех моделей выполнена 10-fold cross-validation. Наиболее стабильной считается модель с минимальным стандартным отклонением R².

### Важное ограничение
Демонстрационный набор из методички содержит всего 20 наблюдений, поэтому 10-fold CV даёт очень маленькие тестовые подвыборки. Полученные оценки могут иметь высокую дисперсию. Для полноценной оценки моделей предпочтителен полный Ames Housing Dataset.